# 03 — Reporting (Local)

Simple visual summary and a "notification payload" preview (Teams/Email/ADO).

In [ ]:
from pathlib import Path
import pandas as pd
import json

BASE = Path(r"/mnt/data/dq_poc_local")
RESULTS_DIR = BASE / "results"

# Pick the latest run results file
files = sorted(RESULTS_DIR.glob("*__rule_results.csv"))
latest = files[-1] if files else None
print("Latest:", latest)
res_df = pd.read_csv(latest) if latest else pd.DataFrame()
display(res_df.head())

## KPI summary

In [ ]:
if not res_df.empty:
    kpi = (res_df.groupby(["dataset_id","severity","status"]).size()
           .reset_index(name="count")
           .sort_values(["dataset_id","severity","status"]))
    display(kpi)

## Top failures

In [ ]:
fails = res_df[res_df["status"]=="fail"].copy()
display(fails[["dataset_id","rule_id","severity","rule_type","sample_ref","observed_value"]].head(50))

## Notification payload preview (what you'd send to Teams/Email/ADO)

In [ ]:
def build_notification_payload(res_df):
    fails = res_df[res_df["status"]=="fail"].copy()
    critical = fails[fails["severity"]=="critical"]
    status = "FAIL" if len(critical)>0 else ("WARN" if len(fails)>0 else "PASS")

    headline = f"[DQ] Local run result: {status} | datasets={res_df['dataset_id'].nunique()} | failed_rules={len(fails)}"
    items = []
    for _, r in fails.head(20).iterrows():
        items.append({
            "dataset_id": r["dataset_id"],
            "rule_id": r["rule_id"],
            "severity": r["severity"],
            "rule_type": r["rule_type"],
            "sample_ref": r.get("sample_ref", None)
        })
    return {"headline": headline, "status": status, "top_failures": items}

payload = build_notification_payload(res_df)
print(json.dumps(payload, indent=2))